Consider this pipeline:
```python
df = load_data()
df = clean_data(df)
df = feature_engineering(df)
train_model(df)
```

If **any assumption is broken,** Pandas will often:
- continue silently
- produce NaNs
- misalign rows
- explode rows
- leak future data

### The Core Idea: *Assumptions -> Assertions*
Every DataFrame has **implicit assumptions**:
- row count
- uniqueness
- nullability
- data types
- value ranges
- relationships

In production, assumptions must be **validated**, not trusted.

### Scehma Validation
#### Check column existence
```python
expected_cols = {"user_id", "age", "salary"}
assert expected_cols.issubset(df.columns)
```

#### Check column order (APIs / ML models)
```python
assert list(df.columns) == [
    "user_id", "age", "salary"
]
```

### Row Count Invariants
#### After joins
```python
assert len(result) >= len(left_df)
```

This catches:
- row explosion
- accidental inner joins
- bad merge keys

### Uniqueness Constraints
#### Primary key check
```python
assert df["user_id"].is_unique
```

#### Composite key check
```python
assert not df.duplicated(
    subset = ["user_id", "date"]
).any()
```

Essential before:
- setting index
- time-series modeling
- joins

### Nullability Rules
#### Column must NOT contain nulls
```python
assert df["user_id"].notna().all()
```

#### Column MAY contain nulls
```python
assert df["salary"].isna().mean() < 0.05
```

### Data Type Enforcement
#### Check dtypes explicitly
```python
assert df["age"].dtype == "Int64"
assert df["salary"].dtype == "float64"
```

#### Datetime correctness
```python
assert pd.api.types.is_datetime64_any_dtype(df["date"])
```

Prevents `.dt` crashes later

### Value Range & Domain Checks
#### Numeric ranges
```python
assert (df["age"].between(0, 120)).all()
assert (df["salary"] >= 0).all()
```

#### Categorical domain
```python
valid_cities = {"Pune", "Mumbai", "Delhi"}
assert set(df["city"].dropna()).issubset(valid_cities)
```

### Time-Series Safety Checks
#### Sorted index
```python
assert df.index.is_monotonic_increasing
```

#### No future leakage
```python
assert df.index.max() <= pd.Timestamp.today()
assert df.index.max() <= pd.Timestamp.utcnow() + pd.Timedelta(minutes=5)
```

#### Continuous frequency (if required)
```python
assert df.index.freq == "D"
```

### Join Safety Checks
#### Use merge indicator
```python
merged = pd.merge(
    users, orders,
    on="user_id",
    how="left",
    indicator=True
)

assert (merged["_merge"] != "right_only").all()
```

#### Validate cardinality
```python
pd.merge(
    users,
    orders,
    on="user_id",
    validate="one_to_many"
)
```

### Feature Engineering Assertions
#### No leakage check
```python
assert (df["lag_1"].isna().sum() > 0)
```

- Lag features *must* have NaNs at start
- If not -> leakage occurred

#### Feature completeness
```python
assert df.dropna().shape[0] > 0.9 * len(df)
```

### Defensive Pandas Patterns
#### Freeze schema early
```python
df = df[EXPECTED_COLUMNS].copy()
```

### Lightweight Validation Function
```python
def validate_users(df):
    assert df["user_id"].is_unique
    assert df["age"].between(0, 120).all()
    assert df["salary"].ge(0).all()
    assert.pd.api.types.is_integer_dtype(df["user_id"])
```

Call it after every major step

### Base DataFrame

In [1]:
import pandas as pd

df = pd.DataFrame({
    "user_id": [101, 102, 102, 104, None],
    "age": [25, -5, 30, 150, 40],
    "city": ["Pune", "Mumbai", "Delhi", "Pune", "Puna"],
    "salary": [50000, 60000, None, -1000, 70000],
    "signup_date": [
        "2024-01-01",
        "2024-01-02",
        "invalid_date",
        "2025-12-01",
        "2024-01-05"
    ]
})

### Exercise 1
1. Assert that the DataFrame contains exactly these columns:
```python
{"user_id", "age", "city", "salary", "signup_date"}
```
2. Assert that no extra columns exist.
3. Explain why column order may matter in ML pipelines.

In [7]:
expected_cols = {"user_id", "age", "city", "salary", "signup_date"}

assert set(df.columns) == expected_cols

- Prevents **schema drift**
- Prevents silent failures when upstream adds/removes columns

In ML pipelines:
- Models expect features in a **fixed order**
- Swapped columns = **wrong predictions without errors**

### Exercise 2
1. Assert:
   - `user_id` has **no missing values**
   - `user_id` is **unique**
2. Print which rows violate uniqueness.
3. Explain why duplicate primary keys are dangerous **even if values look correct.**


In [12]:
assert df["user_id"].notna().all()

AssertionError: 

In [13]:
assert df["user_id"].is_unique

AssertionError: 

In [20]:
df[df["user_id"].duplicated(keep=False)]

,user_id,age,city,salary,signup_date
1,102.0,-5,Mumbai,60000.0,2024-01-02
2,102.0,30,Delhi,NaN,invalid_date


Even if values look correct:
- Joins explode rows
- Aggregations double-count
- ML models overweight entities

### Exercise 3
1. Convert `signup_date` to datetime safely.
2. Assert that `signup_date` is a valid datetime dtype.
3. Count invalid dates.
4. Decide (and justify):
   - Drop invalid rows **or**
   - Block the pipeline with an assertion.

In [21]:
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

In [22]:
assert pd.api.types.is_datetime64_any_dtype(df["signup_date"])

In [23]:
df["signup_date"].isna().sum()

np.int64(1)

Decision: FAIL PIPELINE
- Signup date is a business-critical field
- Guessing or dropping causes data leakage

### Exercise 4
1. Assert:
   - `age` is between `0` and `120`
   - `salary` is **non-negative**
2. Print rows violating these rules.
3. Explain:
   - Why silently clipping values is dangerous.

In [24]:
assert df["age"].between(0, 120).all()

AssertionError: 

In [25]:
assert df["salary"].ge(0).all()

AssertionError: 

In [26]:
df[
    (~df["age"].between(0, 120)) |
    (df["salary"] < 0)
]

,user_id,age,city,salary,signup_date
1,102.0,-5,Mumbai,60000.0,2024-01-02
3,104.0,150,Pune,-1000.0,2025-12-01


Why NOT clip:
- Hides upstream corruption
- Models learn wrong distributions
- Monitoring never triggers

### Exercise 5
1. Allowed cities:
```python
{"Pune", "Mumbai", "Delhi"}
```
2. Assert all `city` values belong to this set.
3. Print invalid values.
4. Explain:
   - Why category drift is a serious production issue.

In [30]:
valid_cities = {"Pune", "Mumbai", "Delhi"}

assert set(df["city"].dropna()).issubset(valid_cities)

AssertionError: 

In [31]:
set(df["city"]) - valid_cities

{'Puna'}

### Exercise 6
1. Assert:
   - `user_id` must NEVER be null
   - `salary` can be null, but < 10%
2. Compute missing ratio for `salary`.
3. Decide:
   - Fail pipeline or continue?

Explain your reasoning.

In [32]:
assert df["user_id"].notna().all()

AssertionError: 

In [33]:
salary_null_ratio = df["salary"].isna().mean()
salary_null_ratio

np.float64(0.2)

**Result:** 0.2 (20%)

In [34]:
assert salary_null_ratio < 0.10

AssertionError: 

Reason:
- Salary is core business signal
- Imputation here = bias + leakage

### Exercise 7
1. Assert:
   - No `signup_date` exists **in the future**
2. Explain:
   - Why this is considered **data leakage.**

In [35]:
assert df["signup_date"].max() <= pd.Timestamp.today()

Future timestamps:
- Encode information unavailable at prediction time
- Inflate model performance
- Fail catastrophically in production

### Exercise 8
Assume you created:
```python
df["salary_log"] = np.log(df["salary"])
```
1. Assert `salary_log` has:
   - No infinite values
   - No unexpected NaNs
2. Explain:
   - Why feature validation is critical **after transformation,** not before.

#### No infinities
```python
assert np.isfinite(df["salary_log"]).all()
```

FAILS — log(0 / NaN / negative)

#### No unexpected NaNs
```python
assert df["salary_log"].notna().all()
```

**Why validate AFTER transformation**
- Transformations introduce new failure modes
- NaNs & infs break models silently
- Earlier validations don’t protect derived features